#  ongoing orders without paid invoices in the current fiscal year 

In [ ]:
import pandas as pd
import requests
from datetime import datetime

pd.set_option('display.max_columns', None)

## 1. Log in from a shared notebook

This is the notebook that will authenticate you to the FOLIO API and store your access token in a variable called ACCESS_TOKEN. You will need to run this cell before running any other cells in this notebook.

In [ ]:
%run folio_auth.ipynb

## Logic
- Get all POs with a status of open and orderType of Ongoing
- Get all POLs with receiptStatus of Ongoing
- Get all Invoices with an Invoice Date of the current FY
- Merge and highlight those POLs with no invoice date or invoice paid date for the current FY. --> This could capture invoices created, but not approved/paid.


In [ ]:
def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit
    return all_records

In [ ]:
fys_raw = fetch_all_records(
    "/finance/fiscal-years",
    records_key="fiscalYears"
) 
fys_df = pd.DataFrame(fys_raw)
today = datetime.today().strftime("%Y-%m-%d")
cur_fy_df = fys_df[(fys_df['periodStart'] <= today) & (fys_df['periodEnd'] >= today)]
styled_fy = cur_fy_df.style.set_properties(
  **{'border': '1px solid black', 'background-color': 'lightgrey'}  
)
print(f"Fiscal Years that encompass {today}")
cur_fy_df[
    ['id', 'name', 'periodStart', 'periodEnd']
].style.set_properties(
    **{'border': '1px solid black'}
)

1. Get all POs ongoing POs with a status of open and orderType of Ongoing
2. Get all POLs with receiptStatus of Ongoing
3. Get all Invoices with an Invoice Date of the current FY
4. Merge and highlight rows with no invoices

In [ ]:
orders_raw = fetch_all_records(
    "/orders/composite-orders",
    records_key="purchaseOrders",
    query='workflowStatus="Open" and orderType = "Ongoing"'
    ) 
orders_df = pd.DataFrame(orders_raw)

print(f"{len(orders_raw)} open, ongoing orders found")
orders_df.head()

In [8]:
poLines_raw = fetch_all_records(
    "/orders/order-lines",
    records_key="poLines",
    query='receiptStatus = "Ongoing"'
    ) 
poLines_df = pd.DataFrame(poLines_raw)

print(f"{len(poLines_raw)} purchase order lines found with a receipt stats of ongoing")
poLines_df.head()

50 purchase order lines found with a receipt stats of ongoing


,id,edition,checkinItems,acquisitionMethod,automaticExport,alerts,claims,claimingActive,claimingInterval,collection,contributors,cost,details,donorOrganizationIds,fundDistribution,instanceId,isPackage,locations,searchLocationIds,orderFormat,paymentStatus,physical,poLineNumber,publicationDate,publisher,purchaseOrderId,receiptStatus,reportingCodes,rush,source,titleOrPackage,vendorDetail,metadata,eresource,requester,cancellationRestriction,cancellationRestrictionNote,description,receiptDate,renewalNote,customFields
0,2de67573-9f06-4819-8c2d-79aa1da01a70,,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],True,30.0,False,[],"{'listUnitPrice': 300.0, 'currency': 'USD', 'd...",{'receivingNote': 'Check for supplementary mat...,[],"[{'code': 'DEMOSCI', 'encumbrance': '4a53cb5d-...",f94bd3f9-01e9-4d1f-a377-36c1227b1168,False,[{'holdingId': '61cf8fdd-4574-4d38-afd5-87d8e1...,[ecc6b76e-4b6a-49de-8378-d7edd8ba3e20],Physical Resource,Ongoing,"{'createInventory': 'Instance, Holding, Item',...",22246-1,,Springer Science+Business Media ; Springer US],bd4c36a7-548e-4278-9530-d404d7c79e46,Ongoing,[],False,User,Journal of prevention.,"{'instructions': '', 'vendorAccount': '97855',...",{'createdDate': '2026-08-02T15:27:44.004+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,046c74a6-6c38-45a3-9235-dd735c3654a4,,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],False,30.0,False,[{'contributor': 'Society for International De...,"{'listUnitPrice': 1362.0, 'currency': 'USD', '...","{'isAcknowledged': False, 'isBinderyActive': T...",[],"[{'code': 'DEMOBUS', 'encumbrance': '0dcf042e-...",514cb527-60f0-4028-b605-e5f39737f544,False,[{'holdingId': '22665aec-9210-451b-a6e4-2c53a6...,[ecc6b76e-4b6a-49de-8378-d7edd8ba3e20],Physical Resource,Ongoing,"{'createInventory': 'Instance, Holding, Item',...",22216-1,[©1978]-,[Society for International Development],75b52c31-b14a-4aa1-90fd-5f333f745fe8,Ongoing,[],False,User,Development = Développement = Desarrollo.,"{'instructions': '', 'vendorAccount': '97855',...",{'createdDate': '2026-06-16T20:13:22.755+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,a0ef81e7-feae-4ad9-a478-b387a1265e46,NaN,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],True,30.0,False,[],"{'listUnitPrice': 100.0, 'currency': 'USD', 'd...","{'receivingNote': 'check for loose pieces', 'i...",[],"[{'code': 'SERIALLAW', 'encumbrance': '70a1f4b...",e94b6fcc-5e53-4084-ad4c-44087842b217,False,[{'holdingId': 'f9a6410a-7481-44ec-a10f-175f4b...,[e3775cef-9d30-4896-a452-98a099cf9bfa],Physical Resource,Ongoing,"{'createInventory': 'Instance, Holding, Item',...",21712-1,NaN,NaN,f74c9128-0666-43c9-8698-0eeea8dfc036,Ongoing,[],False,User,Journal of Aging,"{'instructions': '', 'vendorAccount': '', 'ref...",{'createdDate': '2025-05-29T12:00:20.116+00:00...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ac6455ff-c25a-4131-aa72-69ad98d2142c,NaN,False,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],False,5.0,False,[],"{'listUnitPriceElectronic': 1000.0, 'currency'...","{'isAcknowledged': False, 'isBinderyActive': F...",[],"[{'code': 'DEMOBUS', 'encumbrance': '4ffe53e8-...",33e9764c-c8c8-49fd-87b0-cd363f169e74,False,[{'locationId': 'd0aecd46-fc34-47c2-8702-660b6...,[d0aecd46-fc34-47c2-8702-660b6e904656],Electronic Resource,Ongoing,NaN,22069-1,NaN,NaN,905a3a93-4336-491a-80b8-7fa182caa837,Ongoing,[],False,User,Lexis Nexis Database,"{'instructions': '', 'vendorAccount': '', 'ref...",{'createdDate': '2026-03-05T20:40:29.113+00:00...,"{'activated': False, 'createInventory': 'Insta...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3fa1932c-1250-4851-90cc-fea9424b7a18,NaN,True,28f6e038-52d2-4a04-a14b-8dffedd1d8bb,False,[],[],True,1.0,False,"[{'contributor': 'The Teaching Company', 'cont...","{'listUnitPrice': 21.0, 'currency': 'USD', 'di...","{'receivingNote': '12 issues per year, plus a ...",[],[],84d8a248-60c5-41e7-8b64-f9f2c8b8e981,False,[{'holdingId': 'a2a84370-0b0b-412a-a5be-a831ed...,[ecc6b76e-4b6a-49de-8378-d7edd8ba3e20],Physical Resource,Ongoing,"{'createInventory': 'Instance, Holding, Item'